# Diagnostic recovery — three failed methods

EditGuard: BasicSR dependency and torchvision import compatibility. SepWater: pinned setuptools providing pkg_resources. ChunkySeal: three-hour preparation allowance including checkpoint download.

Only these three methods are retried. The original bank and previous diagnostic folder are read-only. Results go to a new recovery folder. No FIN/HiDDeN or matched-quality changes are applied.

Use the same A100/high-RAM runtime. Upload the accompanying recovery source ZIP when asked. Failed scratch is retained while the runtime survives; completed recovery cases are saved on Drive.

## 1. Paths and scope

Leave N_IMAGES=5 for this check. The same images/messages are read from the existing bank. Changing the diagnostic scope requires a new OUTPUT_ROOT; this is not the production100 run.

In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess, hashlib, zipfile, io

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

N_IMAGES = 5
DEVICE = "cuda"
CHECK_BLOCKED_METHODS = True
DOWNLOAD_AUTHOR_ASSETS = True
MAX_HOURS = 12
SOURCE_ARCHIVE = Path("Embedding_Bank_DIAGNOSTICS_RECOVERY_source.zip")

if IN_COLAB:
    BANK_ROOT = Path("/content/drive/MyDrive/watermarking_embedding_bank_v1")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/watermarking_embedding_diagnostics_recovery_v1")
    SCRATCH_ROOT = Path("/content/watermarking_embedding_diagnostics_recovery_scratch")
else:
    BANK_ROOT = Path("E:/watermarking_embedding_bank_v1")
    OUTPUT_ROOT = Path("E:/watermarking_embedding_diagnostics_recovery_v1")
    SCRATCH_ROOT = Path.home() / "watermarking_embedding_diagnostics_recovery_scratch"

SETTINGS = dict(bank=str(BANK_ROOT), output=str(OUTPUT_ROOT), scratch=str(SCRATCH_ROOT),
                n_images=N_IMAGES, device=DEVICE, check_blocked=CHECK_BLOCKED_METHODS,
                download_author_assets=DOWNLOAD_AUTHOR_ASSETS, max_hours=MAX_HOURS,
                cleanup=True)
print("Read-only bank:", BANK_ROOT)
print("Diagnostic results:", OUTPUT_ROOT)
print("Same historical images and messages. No attacks, training or production edits.")

# Retry only the three technical failures. The eight completed methods stay untouched.
PREVIOUS_DIAGNOSTICS_ROOT = OUTPUT_ROOT.parent / "watermarking_embedding_diagnostics_v1"
SETTINGS.update(models=["editguard", "sepwater", "chunkyseal"], check_blocked=False,
                download_author_assets=False, audit_matching=False, dependency_repairs=True,
                previous_diagnostics=str(PREVIOUS_DIAGNOSTICS_ROOT), keep_failed_scratch=True,
                prepare_timeout_seconds={"chunkyseal": 10800, "editguard": 5400, "sepwater": 5400})
print("Recovery only:", SETTINGS["models"])
print("Previous diagnostic report remains in:", PREVIOUS_DIAGNOSTICS_ROOT)


## 2. Mount Drive and verify the source ZIP

In [ ]:
if IN_COLAB:
    drive.mount("/content/drive")
    if not Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("Drive is not mounted")
if not (BANK_ROOT / "_inputs/manifest.json").is_file():
    raise FileNotFoundError(f"Historical bank not found: {BANK_ROOT}. Set BANK_ROOT to the existing bank, not an empty folder.")
if not shutil.which("git"):
    raise RuntimeError("Git is required for the pinned author repositories")
if DEVICE == "cuda" and not shutil.which("nvidia-smi"):
    raise RuntimeError("Select a GPU runtime before running diagnostics")
for left, right in ((BANK_ROOT, OUTPUT_ROOT), (BANK_ROOT, SCRATCH_ROOT), (OUTPUT_ROOT, SCRATCH_ROOT)):
    if left.resolve().is_relative_to(right.resolve()) or right.resolve().is_relative_to(left.resolve()):
        raise ValueError("Bank, diagnostic results and scratch must be separate, non-nested folders")

SOURCE_SHA256 = "5744031145af7e6324037b263d1cf91c0181920ad7f6f0138150fd2c4f1a7a29"
payload = SOURCE_ARCHIVE.read_bytes() if SOURCE_ARCHIVE.is_file() else None
if IN_COLAB and (payload is None or hashlib.sha256(payload).hexdigest() != SOURCE_SHA256):
    from google.colab import files
    print("Upload Embedding_Bank_DIAGNOSTICS_RECOVERY_source.zip supplied with THIS notebook.")
    uploads = files.upload()
    matching = [data for data in uploads.values() if hashlib.sha256(data).hexdigest() == SOURCE_SHA256]
    if not matching:
        raise ValueError("Source ZIP checksum mismatch")
    payload = matching[0]
if payload is None or hashlib.sha256(payload).hexdigest() != SOURCE_SHA256:
    raise ValueError("Place the matching source ZIP beside the notebook or set SOURCE_ARCHIVE")

PROJECT = SCRATCH_ROOT / "source" / SOURCE_SHA256[:16]
PROJECT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as bundle:
    for item in bundle.infolist():
        target = (PROJECT / item.filename).resolve()
        if not target.is_relative_to(PROJECT.resolve()):
            raise ValueError("Unsafe archive path")
        content = bundle.read(item)
        if target.exists() and target.read_bytes() != content:
            raise ValueError(f"Release file was modified: {target}; use a fresh scratch directory")
        if not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(content)

# Heavy model dependencies never enter the notebook kernel or controller.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv==0.8.15"], check=True)
import uv
UV = str(uv.find_uv_bin())
controller_dir = SCRATCH_ROOT / "controller"
CONTROLLER = controller_dir / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
if not CONTROLLER.exists():
    subprocess.run([UV, "venv", "--python", "3.10", str(controller_dir)], check=True)
subprocess.run([UV, "pip", "install", "--python", str(CONTROLLER), "numpy==1.26.4",
                "Pillow==10.4.0", "psutil==6.1.1", "gdown==5.2.0"], check=True)
ENV = os.environ.copy()
ENV.update(PYTHONPATH=os.pathsep.join((str(PROJECT / "src"), str(PROJECT / "diagnostics"))),
           PYTHONUNBUFFERED="1", OMP_NUM_THREADS="4", OPENBLAS_NUM_THREADS="1",
           PATH=str(Path(UV).parent) + os.pathsep + ENV.get("PATH", ""))
CONFIG = SCRATCH_ROOT / "diagnostic_settings.json"
CONFIG.write_text(json.dumps(SETTINGS, indent=2), encoding="utf-8")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Free scratch SSD: %.1f GiB" % (shutil.disk_usage(SCRATCH_ROOT).free / 1024**3))
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=True)
print("Ready. Stop any older diagnostic run before proceeding.")


## 3. Retry EditGuard, SepWater and ChunkySeal

Do not run the old diagnostic notebook in parallel. The three-hour limit is a safety ceiling, not a runtime estimate.

In [ ]:
# Same cell after a disconnect: measured cases are reused, including nonzero BER.
# Only technical failures/incomplete stages are retried; no new messages are sampled.
command = [str(CONTROLLER), "-m", "bank_diagnostics.controller", "run", "--settings", str(CONFIG)]
process = subprocess.Popen(command, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, encoding="utf-8", errors="replace", start_new_session=(os.name != "nt"))
try:
    for line in process.stdout:
        print(line, end="", flush=True)
    code = process.wait()
    if code:
        raise RuntimeError(f"Diagnostic controller exited with {code}; saved evidence remains in {OUTPUT_ROOT}")
except BaseException:
    if process.poll() is None:
        if os.name != "nt":
            import signal
            os.killpg(process.pid, signal.SIGTERM)
        else:
            process.terminate()
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            process.kill()
    raise
print("Finished or session budget reached. Read REPORT.md; completion does not imply zero BER.")


## 4. Review and export

No correction is applied to the main bank automatically. Review the evidence before changing its adapters or PSNR policy.

In [ ]:
subprocess.run([str(CONTROLLER), "-m", "bank_diagnostics.controller", "report",
                "--settings", str(CONFIG)], env=ENV, check=True)
from IPython.display import Markdown, display
display(Markdown((OUTPUT_ROOT / "REPORT.md").read_text(encoding="utf-8")))
print("Full stage results:", OUTPUT_ROOT / "stages.csv")
print("Previous PSNR audit:", PREVIOUS_DIAGNOSTICS_ROOT / "matching/summary.json")
print("Reused author weights:", OUTPUT_ROOT / "author_assets")

# A small review package: logs and JSON/CSV/Markdown, not large weights or image copies.
archive_path = OUTPUT_ROOT / "diagnostics_report.zip"
with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as bundle:
    for p in OUTPUT_ROOT.rglob("*"):
        if p.is_file() and p.suffix.lower() in {".json", ".csv", ".md", ".log", ".lock", ".txt"}:
            bundle.write(p, p.relative_to(OUTPUT_ROOT).as_posix())
print("Send this file for review:", archive_path)
